# Exporta.co — Pipeline V3 (Colab, bajo consumo de RAM)
**Estrategia:** procesa un año a la vez, guarda Parquet inmediatamente y libera RAM antes de continuar.  
**Migración local:** cambiar `MODO = 'local'` en S1 y ajustar rutas.  
**Ejecucion rapida:** una vez generados los Parquet, comentar S5 y cargar directamente desde S6.

---
## S1 — Setup: instalacion, imports y rutas

In [ ]:
# Instalaciones (solo necesarias en Colab)
import subprocess, sys
for pkg in ['pycountry', 'pyarrow']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import pandas as pd
import numpy as np
import json, zipfile, io, re, gc
from pathlib import Path
from collections import defaultdict
import pycountry

print(f'pandas {pd.__version__} | numpy {np.__version__}')

In [ ]:
MODO = 'colab'  # cambiar a 'local' para ejecucion local

if MODO == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

    import subprocess
    REPO_DIR = Path('/content/Exporta_Co')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/CamiloJose90/Exporta_Co.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'])

    DRIVE_DIR  = Path('/content/drive/MyDrive/exporta_co')
    RAW_DIR    = DRIVE_DIR / 'raw'
    CLEAN_DIR  = DRIVE_DIR / 'clean'
    OUTPUT_DIR = REPO_DIR / 'data'

else:
    RAW_DIR    = Path(r'C:\Users\camil\OneDrive\Documentos\MIAD\Visualización y storytelling\Proyecto')
    CLEAN_DIR  = RAW_DIR / 'clean'
    OUTPUT_DIR = RAW_DIR / 'json_output'

CLEAN_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

assert RAW_DIR.exists(), f'RAW_DIR no encontrado: {RAW_DIR}'
print(f'RAW_DIR   : {RAW_DIR}')
print(f'CLEAN_DIR : {CLEAN_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

---
## S2 — Configuracion
Ajustar `AÑOS` para ampliar o reducir el rango. Si el DANE cambia columnas, editar los mapas aqui.

In [ ]:
AÑOS = list(range(2011, 2026))

COLUMNAS_EXPO = {
    'ï»¿fech': 'fech', 'fech': 'fech',
    'cod_pai4': 'pais',       # ISO alfa-3 directo (USA, DEU...)
    'posar':    'subpartida', # posicion arancelaria HS
    'fobdol':   'fob_usd',
    'pnk':      'kg_neto',
}

COLUMNAS_IMPO = {
    'ï»¿fech': 'fech', 'fech': 'fech',
    'paisgen':  'pais',       # codigo numerico ISO 3166-1
    'naban':    'subpartida',
    'vacid':    'cif_usd',    # valor CIF dolares
    'pnk':      'kg_neto',
}

# Mes en espanol -> numero (extrae mes desde nombre de archivo IMPO)
MESES_ES = {
    'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
    'julio':7,'agosto':8,'septiembre':9,'octubre':10,'noviembre':11,'diciembre':12,
}

# Sectores OMC desde capitulo HS (primeros 2 digitos de posicion arancelaria)
_SECTOR_BREAKS  = [0, 24, 27, 97]
_SECTOR_LABELS  = ['Agropecuarios, alimentos y bebidas',
                   'Combustibles e industrias extractivas',
                   'Manufacturas']

print('Configuracion lista')

---
## S3 — Funciones auxiliares

In [ ]:
def normalizar_columnas(df, mapa):
    rename = {}
    for col in df.columns:
        clave = col.strip().lower().replace(' ', '_')
        if clave in mapa:      rename[col] = mapa[clave]
        elif col.strip() in mapa: rename[col] = mapa[col.strip()]
    return df.rename(columns=rename)


def limpiar_valor_monetario(serie):
    """Formato europeo: punto=miles, coma=decimal (ej: 4.885,70 -> 4885.70)."""
    return (serie.astype(str).str.strip()
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .pipe(pd.to_numeric, errors='coerce')
            .astype('float32'))


def derivar_sector(serie_subpartida):
    """Sector OMC desde primeros 2 digitos de la posicion arancelaria (vectorizado)."""
    cap = pd.to_numeric(serie_subpartida.astype(str).str[:2], errors='coerce')
    return (pd.cut(cap, bins=_SECTOR_BREAKS, labels=_SECTOR_LABELS)
            .astype(str).replace('nan', 'Otros sectores').astype('category'))


def mes_desde_nombre(nombre_archivo):
    """Extrae numero de mes desde nombre del archivo CSV (ej: 'Agosto.csv' -> 8)."""
    return MESES_ES.get(Path(nombre_archivo).stem.lower().strip(), None)


def leer_csv_bytes(data_bytes):
    """Lee CSV desde bytes probando encodings y separadores comunes del DANE."""
    for encoding in ('latin-1', 'utf-8', 'utf-8-sig'):
        for sep in (',', ';', '\t'):
            try:
                df = pd.read_csv(io.BytesIO(data_bytes), sep=sep, encoding=encoding,
                                 low_memory=False, on_bad_lines='skip')
                if len(df.columns) > 2:
                    return df.dropna(how='all')
            except Exception:
                continue
    return None


_cache_iso3 = {}
def codigo_numerico_a_iso3(codigo):
    """Convierte codigo numerico ISO 3166-1 a ISO alfa-3 (con cache)."""
    if pd.isna(codigo): return None
    key = str(int(codigo)).zfill(3)
    if key not in _cache_iso3:
        try:    _cache_iso3[key] = pycountry.countries.get(numeric=key).alpha_3
        except: _cache_iso3[key] = None
    return _cache_iso3[key]


print('Funciones listas')

---
## S4 — Descubrimiento de archivos
Verificar que todos los ZIPs esten correctamente ubicados antes de procesar.

In [ ]:
PATRON = re.compile(r'^(expo|impo)_(\d{4})(?:_(\d))?\.zip$', re.IGNORECASE)
archivos = {'expo': defaultdict(list), 'impo': defaultdict(list)}

for f in sorted(RAW_DIR.glob('*.zip')):
    m = PATRON.match(f.name)
    if m:
        archivos[m.group(1).lower()][int(m.group(2))].append(f)

for tipo in ('expo', 'impo'):
    en_rango   = sorted(a for a in archivos[tipo] if a in AÑOS)
    faltantes  = [a for a in AÑOS if a not in archivos[tipo]]
    print(f'\n{tipo.upper()} — {len(en_rango)} años encontrados')
    for año in en_rango:
        partes = archivos[tipo][año]
        tag = f'[{len(partes)} partes]' if len(partes) > 1 else ''
        for p in partes: print(f'  {p.name} {tag}')
    if faltantes: print(f'  Faltantes: {faltantes}')

---
## S5 — Procesamiento año a año con liberacion de RAM
Cada año se procesa, limpia y guarda como Parquet de forma independiente.  
La RAM se libera despues de cada año con `gc.collect()`.  
**Para omitir este paso en ejecuciones posteriores:** comentar la llamada a `procesar_todo()` y cargar desde S6.

In [ ]:
def procesar_zip_expo(ruta_zip):
    """Lee ZIP EXPO: ZIP anual > ZIPs mensuales > CSV."""
    frames = []
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            zips_mes = sorted(f for f in zf.namelist() if f.lower().endswith('.zip'))
            for nom in zips_mes:
                try:
                    with zipfile.ZipFile(io.BytesIO(zf.read(nom)), 'r') as zf_mes:
                        for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                            df = leer_csv_bytes(zf_mes.read(csv_path))
                            if df is not None:
                                frames.append(normalizar_columnas(df, COLUMNAS_EXPO))
                except Exception as e:
                    print(f'    Advertencia {nom}: {e}')
    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


def procesar_zip_impo(ruta_zip):
    """
    Lee ZIP IMPO con dos casos:
    - Caso A (2011-2023, 2025): ZIP anual > ZIPs mensuales > CSV
    - Caso B (2024): ZIP anual > carpetas > CSV directos
    """
    frames = []
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            contenidos  = zf.namelist()
            zips_mes    = sorted(f for f in contenidos if f.lower().endswith('.zip'))
            csvs_dir    = sorted(f for f in contenidos if f.lower().endswith('.csv'))

            if zips_mes:  # Caso A
                for nom in zips_mes:
                    try:
                        with zipfile.ZipFile(io.BytesIO(zf.read(nom)), 'r') as zf_mes:
                            for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                                df = leer_csv_bytes(zf_mes.read(csv_path))
                                if df is not None:
                                    df = normalizar_columnas(df, COLUMNAS_IMPO)
                                    mes = mes_desde_nombre(csv_path)
                                    if mes: df['mes'] = mes
                                    frames.append(df)
                    except Exception as e:
                        print(f'    Advertencia {nom}: {e}')

            elif csvs_dir:  # Caso B
                for csv_path in csvs_dir:
                    try:
                        df = leer_csv_bytes(zf.read(csv_path))
                        if df is not None:
                            df = normalizar_columnas(df, COLUMNAS_IMPO)
                            mes = mes_desde_nombre(csv_path)
                            if mes: df['mes'] = mes
                            frames.append(df)
                    except Exception as e:
                        print(f'    Advertencia {csv_path}: {e}')

    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


def limpiar_expo(df, año):
    fech = pd.to_numeric(df['fech'], errors='coerce')
    df['año'] = año
    df['mes'] = (fech % 100).astype('uint8')
    df['iso3'] = df['pais'].str.strip().str.upper().astype('category')
    df['fob_usd'] = limpiar_valor_monetario(df['fob_usd'])
    df['sector'] = derivar_sector(df['subpartida'])
    return df[
        df['fob_usd'].notna() & (df['fob_usd'] > 0) &
        df['iso3'].notna() & (df['iso3'] != '') &
        df['mes'].between(1, 12)
    ][['año', 'mes', 'iso3', 'fob_usd', 'sector']]


def limpiar_impo(df, año):
    fech = pd.to_numeric(df['fech'], errors='coerce')
    df['año'] = año
    if 'mes' not in df.columns:
        df['mes'] = (fech % 100)
    df['mes'] = pd.to_numeric(df['mes'], errors='coerce').astype('uint8')
    df['cif_usd'] = (df['cif_usd'].astype(str).str.strip()
                     .str.replace(',', '.', regex=False)
                     .pipe(pd.to_numeric, errors='coerce')
                     .astype('float32'))
    codigo = pd.to_numeric(df['pais'], errors='coerce')
    df['iso3'] = codigo.map(codigo_numerico_a_iso3).astype('category')
    return df[
        df['cif_usd'].notna() & (df['cif_usd'] > 0) &
        df['iso3'].notna() &
        df['mes'].between(1, 12)
    ][['año', 'mes', 'iso3', 'cif_usd']]


def procesar_todo():
    for tipo, fn_zip, fn_limpiar in [
        ('expo', procesar_zip_expo, limpiar_expo),
        ('impo', procesar_zip_impo, limpiar_impo),
    ]:
        print(f'\nProcesando {tipo.upper()}...')
        for año in AÑOS:
            parquet_path = CLEAN_DIR / f'{tipo}_{año}.parquet'
            if parquet_path.exists():
                print(f'  {año}: ya existe, omitido')
                continue

            partes = archivos[tipo].get(año, [])
            if not partes:
                print(f'  {año}: ZIP no encontrado')
                continue

            frames = []
            for ruta in sorted(partes):
                frames.extend(fn_zip(ruta))

            if not frames:
                print(f'  {año}: sin datos legibles')
                continue

            df = pd.concat(frames, ignore_index=True)
            df = fn_limpiar(df, año)
            df.to_parquet(parquet_path, index=False)
            n = len(df)
            del df, frames
            gc.collect()
            print(f'  {año}: {n:,} filas guardadas')

procesar_todo()

---
## S6 — Carga desde Parquet
Punto de entrada para ejecuciones posteriores cuando los Parquet ya existen.  
En primera ejecucion S5 los genera; aqui solo se cargan.

In [ ]:
df_expo = pd.concat(
    [pd.read_parquet(f) for f in sorted(CLEAN_DIR.glob('expo_*.parquet'))],
    ignore_index=True
)
df_impo = pd.concat(
    [pd.read_parquet(f) for f in sorted(CLEAN_DIR.glob('impo_*.parquet'))],
    ignore_index=True
)
print(f'Expo: {len(df_expo):,} filas | {df_expo["año"].nunique()} años')
print(f'Impo: {len(df_impo):,} filas | {df_impo["año"].nunique()} años')

---
## S7 — Generacion de JSON para D3.js

In [ ]:
# VIZ 1 — Mapa coropletico: FOB por pais destino y año
viz1 = (
    df_expo
    .groupby(['año', 'iso3'], as_index=False)['fob_usd']
    .sum()
    .assign(fob_musd=lambda d: (d['fob_usd'] / 1e6).round(2))
    .drop(columns='fob_usd')
    .sort_values(['año', 'fob_musd'], ascending=[True, False])
)
(OUTPUT_DIR / 'viz1_mapa_destinos.json').write_text(
    json.dumps(viz1.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz1: {len(viz1):,} registros | {viz1["iso3"].nunique()} paises')

In [ ]:
# VIZ 2 — Balanza comercial mensual
expo_m = df_expo.groupby(['año','mes'])['fob_usd'].sum().reset_index(name='expo_usd')
impo_m = df_impo.groupby(['año','mes'])['cif_usd'].sum().reset_index(name='impo_usd')
viz2 = (
    pd.merge(expo_m, impo_m, on=['año','mes'], how='outer').fillna(0)
    .assign(
        periodo            = lambda d: d['año'].astype(str) + '-' + d['mes'].astype(str).str.zfill(2),
        exportaciones_musd = lambda d: (d['expo_usd'] / 1e6).round(2),
        importaciones_musd = lambda d: (d['impo_usd'] / 1e6).round(2),
        balanza_musd       = lambda d: ((d['expo_usd'] - d['impo_usd']) / 1e6).round(2),
    )
    .sort_values('periodo')
    [['periodo','año','mes','exportaciones_musd','importaciones_musd','balanza_musd']]
)
(OUTPUT_DIR / 'viz2_balanza_temporal.json').write_text(
    json.dumps(viz2.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz2: {len(viz2):,} periodos ({viz2["periodo"].min()} -> {viz2["periodo"].max()})')

In [ ]:
# VIZ 3 — Treemap sectores OMC
sector_anual = (
    df_expo.groupby(['año','sector'])['fob_usd'].sum().reset_index()
    .assign(fob_musd=lambda d: (d['fob_usd'] / 1e6).round(2))
)
total_año = sector_anual.groupby('año')['fob_musd'].transform('sum')
sector_anual['pct'] = (sector_anual['fob_musd'] / total_año * 100).round(1)

años_disp = sorted(sector_anual['año'].unique().tolist())
viz3 = {
    'años_disponibles': años_disp,
    'datos_por_año': {
        str(año): {
            'name': f'Exportaciones {año}',
            'children': [
                {'name': r['sector'], 'value': r['fob_musd'], 'pct': r['pct']}
                for _, r in g.sort_values('fob_musd', ascending=False).iterrows()
            ]
        }
        for año, g in sector_anual.groupby('año')
    }
}
(OUTPUT_DIR / 'viz3_treemap_sectores.json').write_text(
    json.dumps(viz3, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'viz3: {len(años_disp)} años | sectores: {sector_anual["sector"].nunique()}')

---
## S8 — Push a GitHub
Solo ejecutar en Colab. Requiere token de acceso personal en `GITHUB_TOKEN`.  
El token no debe quedar guardado en el notebook antes de hacer commit.

In [ ]:
if MODO == 'colab':
    GITHUB_TOKEN  = ''  # pegar token aqui — no commitear con el token visible
    REPO_URL_AUTH = f'https://{GITHUB_TOKEN}@github.com/CamiloJose90/Exporta_Co.git'

    import subprocess
    cmds = [
        ['git', '-C', str(REPO_DIR), 'config', 'user.email', 'pipeline@exporta.co'],
        ['git', '-C', str(REPO_DIR), 'config', 'user.name', 'Exporta Pipeline'],
        ['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL_AUTH],
        ['git', '-C', str(REPO_DIR), 'add', 'data/'],
        ['git', '-C', str(REPO_DIR), 'commit', '-m', 'data: actualizar JSONs desde pipeline'],
        ['git', '-C', str(REPO_DIR), 'push'],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'Error: {r.stderr}'); break
    else:
        print('JSONs pusheados a GitHub')
else:
    print('Modo local: copiar manualmente los JSON de OUTPUT_DIR al repositorio.')